# agri-drone — Notebook 2: PDT calibration + few-shot

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashut0sh-mishra/agri-drone/blob/research-upgrade/notebooks/colab/02_pdt_calibration.ipynb)

Rescues the degenerate v3 PDT result. Writes `results_v2/pdt/PDT_SECTION.md` for v4 §5.4.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d agri-drone || git clone https://github.com/Ashut0sh-mishra/agri-drone.git
%cd /content/agri-drone
!git fetch --all && git checkout research-upgrade && git pull
!pip install -q -r requirements.txt scikit-learn matplotlib


In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU runtime'
DRIVE = '/content/drive/MyDrive/agri-drone'
WEIGHTS = f'{DRIVE}/models/wheat_4cls.pt'
PDT_DIR = f'{DRIVE}/data/PDT_datasets'
OUT = f'{DRIVE}/results_v2/pdt'
import os; os.makedirs(OUT, exist_ok=True)
print('weights:', WEIGHTS); print('pdt:', PDT_DIR); print('out:', OUT)


## 1. Threshold sweep (ROC / PR)


In [ ]:
import pathlib
csv_path = pathlib.Path('evaluate/results/cross_dataset_PDT_predictions.csv')
if not csv_path.exists():
    print('Run evaluate/pdt_cross_eval.py first')
!python evaluate/pdt_v2.py --variant threshold_sweep --predictions-csv $csv_path --out-dir $OUT


In [ ]:
import json, matplotlib.pyplot as plt
r = json.loads(open(f'{OUT}/threshold_sweep.json').read()); print(r)
plt.figure(); plt.title('ROC / PR summary')
plt.bar(['ROC-AUC','PR-AUC'], [r.get('roc_auc',0), r.get('pr_auc',0)])
plt.ylim(0,1); plt.savefig(f'{OUT}/roc_pr_summary.png'); plt.show()


## 2. Few-shot fine-tune (5 / 10 / 25 / 50 per class)


In [ ]:
for k in [5, 10, 25, 50]:
    !python evaluate/pdt_v2.py --variant few_shot --shots $k --out-dir $OUT


## 3. Temperature + Platt scaling


In [ ]:
!python evaluate/pdt_v2.py --variant calibration --out-dir $OUT


## 4. Reliability diagram (placeholder)


In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.figure(); plt.plot([0,1],[0,1],'--',label='perfect')
plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.legend()
plt.title('Reliability (placeholder)'); plt.savefig(f'{OUT}/reliability.png'); plt.show()


## 5. Write PDT_SECTION.md


In [ ]:
import json, pathlib
ts = json.loads(open(f'{OUT}/threshold_sweep.json').read())
md = f'''# §5.4 PDT cross-dataset evaluation (rescued)

- ROC-AUC: {ts.get("roc_auc","[TBD]")}
- PR-AUC: {ts.get("pr_auc","[TBD]")}
- Specificity @ 90% sensitivity: {ts.get("specificity_at_90_sensitivity","[TBD]")}
- Sensitivity @ 90% specificity: {ts.get("sensitivity_at_90_specificity","[TBD]")}

Few-shot and calibration results: see `results_v2/pdt/*.json`.
'''
pathlib.Path(f'{OUT}/PDT_SECTION.md').write_text(md); print(md)
